## Configure API KEYS

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter GOOGLE_API_KEY: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("GOOGLE_API_KEY configured:", bool(os.getenv("GOOGLE_API_KEY")))
print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

# print("GOOGLE_API_KEY configured:", os.getenv("GOOGLE_API_KEY"))
# print("GROQ_API_KEY configured:", os.getenv("GROQ_API_KEY"))
# print("TAVILY_API_KEY configured:", os.getenv("TAVILY_API_KEY"))
# print("PINECONE_API_KEY configured:", os.getenv("PINECONE_API_KEY"))

GOOGLE_API_KEY configured: True
GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


### 1. Load the Document 

In [3]:
from langchain_community.document_loaders import WebBaseLoader

source_url = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=source_url,
    requests_kwargs = {
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    }
)
raw_docs = loader.load()

print(f"Loaded document: {len(raw_docs)}\n documents from: {source_url}")
print("Source: ", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1000])

C:\Users\PRARINAM DERIA\AppData\Local\Temp\ipykernel_8112\3264026688.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded document: 1
 documents from: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Source:  https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase stu

### 2. Split the document into Chunk

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    add_start_index = True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 27

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCre


### Create Embeddings Using Gemini Model

In [5]:
from google import genai
from google.genai import types
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

embeddings = gemini_client.models.embed_content(
    model="gemini-embedding-2",
    contents="What is agentic RAG in LangChain and how does it work?",
    config = types.EmbedContentConfig(
        output_dimensionality=768
    )
)


print("Gemini Embedding Result: ", embeddings)

print("Gemini Embedding Result: ", embeddings.embeddings)
[embedding_obj] = embeddings.embeddings
embedding_length = len(embedding_obj.values)
print("Gemini Embedding Result Length: ", embedding_length)

Gemini Embedding Result:  sdk_http_response=HttpResponse(
  headers=<dict len=11>
) embeddings=[ContentEmbedding(
  values=[
    0.01283944,
    0.013940322,
    0.02108872,
    0.027043901,
    -0.007166158,
    <... 763 more items ...>,
  ]
)] metadata=None
Gemini Embedding Result:  [ContentEmbedding(
  values=[
    0.01283944,
    0.013940322,
    0.02108872,
    0.027043901,
    -0.007166158,
    <... 763 more items ...>,
  ]
)]
Gemini Embedding Result Length:  768


### 3. Custom Gemini Embeddings Class

In [6]:
import os
from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings


# Custom gemini embeddings object
class GeminiSDKEmbeddings(Embeddings):

  def __init__(self, model: str = "gemini-embedding-2", output_dim: int = 768):
    self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    self.model = model
    self.config = types.EmbedContentConfig(output_dimensionality=output_dim)

  def embed_documents(self, texts: list[str]) -> list[list[float]]:
    response = self.client.models.embed_content(
        model=self.model, contents=texts, config=self.config
    )
    return [e.values for e in response.embeddings]

  def embed_query(self, text: str) -> list[float]:
    response = self.client.models.embed_content(
        model=self.model, contents=text, config=self.config
    )
    return response.embeddings[0].values


# Instantiate wrapper
gemini_embeddings = GeminiSDKEmbeddings(
    model="gemini-embedding-2", output_dim=768
)


### 4. Create the Pinecone Vector Database

In [7]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "uniquery-agentic-rag-kb"
NAMESPACE = "uniquery-langgraph-agentic-rag"

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

existing_indexes = [index["name"] for index in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

    # wait until pinecone reports new index is ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(5)



print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=gemini_embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE
    }
)
print("Vector store created and retriever initialized.")
print("Retriever search_kwargs:", retriever)

Pinecone index ready: uniquery-agentic-rag-kb
Vector store created and retriever initialized.
Retriever search_kwargs: tags=['PineconeVectorStore', 'GeminiSDKEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001F34E803E00> search_kwargs={'k': 4, 'namespace': 'uniquery-langgraph-agentic-rag'}


In [10]:
## Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "uniquery-agentic-rag-kb"
NAMESPACE = "uniquery-langgraph-agentic-rag"

pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))

# Load Index
index = pc.Index(INDEX_NAME)

# Connect Existing index with lagchain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=gemini_embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE
    }
)
print("Pinecone retriever loaded:", retriever)
print("Existing Pinecone index loaded and retriever initialized.")

Pinecone retriever loaded: tags=['PineconeVectorStore', 'GeminiSDKEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001F34E8E8770> search_kwargs={'k': 4, 'namespace': 'uniquery-langgraph-agentic-rag'}
Existing Pinecone index loaded and retriever initialized.


### Test Private KB Retrieval

In [12]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, docs in enumerate(kb_docs):
    print(f"\nRetrieved Document {i+1}:")
    print("Source:", docs.metadata.get("source"))
    print("Content Preview:", docs.page_content[:500], "...\n")


Retrieved Document 1:
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Content Preview: Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegration ...

